In [10]:
import json
import os

In [11]:
DATASET_FOLDER = "Datasets"

In [12]:
info_list = []

for item in os.scandir(DATASET_FOLDER):
    if item.is_dir():
        # is folder
        folder_name = item.name
        
        # check if last token is int
        if folder_name.split("_")[0] == "tabular" or folder_name.split("_")[0] == "image":
            # site folder
            # print("Valid folder:", item.path)

            # Look for JSON files inside
            for sub in os.scandir(item.path):
                if sub.is_file() and sub.name.startswith(folder_name) and sub.name.endswith("stats.json"):
                    json_file = sub.path
                    # print("  Found JSON:", json_file)

                    # Load it
                    with open(json_file, "r") as f:
                        data = json.load(f)
                    info_list.append(data)

info_list

[{'site_name': 'tabular_1',
  'n_samples': 229712,
  'feature_means': {'Age': 8.085981576931113,
   'Sex': 0.4391411854844327,
   'BMI': 28.68571080309257,
   'GenHlth': 2.6012049871142997,
   'HighBP': 0.4543776554990597,
   'DiffWalk': 0.1855584383924218,
   'HighChol': 0.441696559169743,
   'HeartDiseaseorAttack': 0.10321620115623041},
  'feature_vars': {'Age': 9.5726421687767,
   'Sex': 0.2462962046957598,
   'BMI': 46.064766216567136,
   'GenHlth': 1.1337465454985667,
   'HighBP': 0.24791860168223753,
   'DiffWalk': 0.15112650433378755,
   'HighChol': 0.24660070878735263,
   'HeartDiseaseorAttack': 0.09256261697510698}},
 {'site_name': 'tabular_3',
  'n_samples': 768,
  'feature_means': {'Age': 33.240885416666664,
   'Sex': 0.0,
   'BMI': 31.992578124999998,
   'GenHlth': 2.5494791666666665,
   'HighBP': 0.078125,
   'DiffWalk': 0.0,
   'HighChol': 0.0,
   'HeartDiseaseorAttack': 0.0},
  'feature_vars': {'Age': 138.12296379937067,
   'Sex': 0.0,
   'BMI': 62.079046478271486,
   'G

In [13]:
len(info_list)

4

In [14]:
from collections import defaultdict
import numpy as np

site_stats = info_list

In [15]:
# ---- define the global feature template ----
feature_template = [
    "Age", "Sex", "BMI", "GenHlth",
    "HighBP", "DiffWalk", "HighChol", "HeartDiseaseorAttack"
]

In [16]:
# ---- compute global means ----
sum_mu = defaultdict(float)
sum_n  = defaultdict(int)

for site in site_stats:
    n = int(site["n_samples"])
    for f in feature_template:
        if f in site["feature_means"]:
            sum_mu[f] += n * site["feature_means"][f]
            sum_n[f]  += n

global_mean = {
    f: (sum_mu[f] / sum_n[f]) if sum_n[f] > 0 else 0.0
    for f in feature_template
}

# ---- compute global variances ----
sum_var = defaultdict(float)

for site in site_stats:
    n = int(site["n_samples"])
    for f in feature_template:
        if f in site["feature_means"]:
            mu_i = site["feature_means"][f]
            var_i = site["feature_vars"][f]
            sum_var[f] += n * (var_i + (mu_i - global_mean[f])**2)

global_var = {
    f: (sum_var[f] / sum_n[f]) if sum_n[f] > 0 else 0.0
    for f in feature_template
}

# ---- compute std ----
global_std = {f: float(np.sqrt(v)) for f, v in global_var.items()}

# ---- final dict ----
global_stats = {
    "global_feature_mean": global_mean,
    "global_feature_std": global_std
}

In [17]:
global_stats

{'global_feature_mean': {'Age': 13.882173297668496,
  'Sex': 0.4350631939078877,
  'BMI': 28.32551647452812,
  'GenHlth': 2.1166934744265036,
  'HighBP': 0.3780250682442712,
  'DiffWalk': 0.14608202095944292,
  'HighChol': 0.3577913856509256,
  'HeartDiseaseorAttack': 0.0816266831206661},
 'global_feature_std': {'Age': 16.049739863110435,
  'Sex': 0.4957656972915252,
  'BMI': 6.703381964565002,
  'GenHlth': 1.3672720127520293,
  'HighBP': 0.4848940446802385,
  'DiffWalk': 0.35318842578975856,
  'HighChol': 0.4793502998902955,
  'HeartDiseaseorAttack': 0.27379511997730066}}

In [18]:
# save to file
with open("global_tabular_stats.json", "w", encoding="utf-8") as f:
    json.dump(global_stats, f, indent=2, ensure_ascii=False)